# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id` values, and fields. All entity references are by their `@id` as specified in the Croissant schema.

Let's list all record sets and their fields:

In [ ]:
# Explore record sets and their fields
record_set_ids = []
print("Available Record Sets and their fields:\n")
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()
# Preview the first few records from the first record set (if available)
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"\nSample records from record set: {first_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:  # Show only first 3
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
We'll extract all available record sets and put them in DataFrames, referencing all record sets by their `@id`.

In [ ]:
# Extract data from all record sets, store them in a dictionary keyed by @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records for RecordSet @id: {record_set_id}")

# Display columns for the main (first) record set
if record_set_ids:
    print('\nColumns for main RecordSet:')
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps (e.g., filtering by numeric field, normalization, grouping), referencing all entities by their `@id`.

Let's select a numeric field for demonstration. You may adjust the selected field by its `@id` if desired.

In [ ]:
# EDA: Select a numeric field by its @id

# For demonstration, let's look for an example numeric field (commonly 'molecular_weight', 'peptide_count', etc.)
field_candidates = list(dataframes[first_record_set_id].select_dtypes(include='number').columns)
if field_candidates:
    numeric_field_id = field_candidates[0]  # Select the first numeric field
    print(f"Using numeric field for filtering and normalization: {numeric_field_id}")
    threshold = dataframes[first_record_set_id][numeric_field_id].mean()
    filtered_df = dataframes[first_record_set_id][dataframes[first_record_set_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try grouping by a categorical field (e.g., the first non-numeric field)
    group_candidates = [c for c in dataframes[first_record_set_id].columns if c != numeric_field_id]
    for group_field_id in group_candidates:
        if pd.api.types.is_string_dtype(dataframes[first_record_set_id][group_field_id]):
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
            break
else:
    print("No numeric fields found to demonstrate EDA.")

## 5. Visualization
Visualize data distributions or relationships. Here's a histogram and boxplot for the selected numeric field. You can extend this with more plots as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if field_candidates:
    plt.figure(figsize=(12, 5))
    plt.subplot(1,2,1)
    sns.histplot(dataframes[first_record_set_id][numeric_field_id].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field_id}")

    plt.subplot(1,2,2)
    sns.boxplot(x=dataframes[first_record_set_id][numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")

    plt.tight_layout()
    plt.show()
else:
    print("No numeric fields found to plot.")

## 6. Conclusion
We successfully loaded the FAIR^2 dataset using its Croissant schema, explored the available record sets and fields by their `@id`, extracted data into DataFrames, performed basic filtering and normalization using only field `@id` references, and visualized numeric data distributions. Further analysis is possible by continuing with field and record set references by their Croissant `@id`.